# Classification: Static Features

Session-level means of all eye-tracking metrics are used as features to predict depression severity. Each metric is averaged across stimulus scenes in a session, with scenes stratified by valence pair type. Features are then averaged per user to reduce within-subject measurement noise.

Three tasks are tested:
- binary classification (depressed vs not)
- multi-class (severity groups; PHQ-9: 5 classes, BDI-II: 4 classes)
- regression (predict score directly)

Every task is run on two depression scales:
- **PHQ-9** (Patient Health Questionnaire, 9 items, range 0–27)
- **BDI-II** (Beck Depression Inventory, 21 items, range 0–63)

Running both scales provides a convergent-validity check.

Three models are compared:
- Logistic / Ridge Regression
- Random Forest
- XGBoost

Across two feature sets:
- all features
- theory-driven

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

from src.features.session_aggregation import build_static_aggregation
from src.evaluation.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance, build_comparison_table
)
from src.visualization.io import save_figure

## 1. Build session-level dataset

### 1.1 Filter scene-level data

Three filters are applied before aggregation:

- `scene_type == "stimulus"` - include only stimulus scenes and exclude fixation_cross scenes
- `scene_quality_valid == True` - standard eye-tracker quality gate applied by pipeline 04
- `scene_valence_pair` in `{negative_vs_positive, negative_vs_neutral, neutral_vs_positive}` - the three meaningful pair types
- `duration_ms` in [500, 10000] ms - exclude tracker outliers (scenes shorter than 0.5s or longer than 10s)

These threshold values are determined by the experimental setup.

In [0]:
VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

scene_metrics = spark.table("anima.scene_metrics")
forms = spark.table("anima.forms")

stimulus_scenes = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

print(f"Clean stimulus scenes: {stimulus_scenes.count()}")

### 1.2 Build aggregated features

In [0]:
agg = build_static_aggregation()
print(f"Aggregation expressions: {len(agg.exprs)}")

session_metrics = stimulus_scenes.groupBy("session_id").agg(*agg.exprs)

df_joined = session_metrics.join(
    forms.select("session_id", "uid", "phq9_score", "phq9_severity", "bdi_score", "bdi_severity"),
    on="session_id",
    how="inner",
)

df = df_joined.toPandas()
print(f"Data: {len(df)} sessions, {df.shape[1]} columns")

### 1.3 Aggregate to user level

In [0]:
FEATURE_COLS = agg.all_columns
NUMERIC_COLS = FEATURE_COLS + ["phq9_score", "bdi_score"]
 
df = df.groupby("uid", as_index=False)[NUMERIC_COLS].mean()
 
print(f"After per-user aggregation: {len(df)} users, {len(FEATURE_COLS)} features")
print(f"PHQ-9: min={df['phq9_score'].min():.1f}, max={df['phq9_score'].max():.1f}, mean={df['phq9_score'].mean():.1f}")
print(f"BDI: min={df['bdi_score'].min():.1f}, max={df['bdi_score'].max():.1f}, mean={df['bdi_score'].mean():.1f}")

## 2. Define targets and feature sets

### 2.1 PHQ-9 targets

Binary cutoff: PHQ-9 ≥ 10.

Severity classes:
- 0 = minimal (0–4)
- 1 = mild (5–9)
- 2 = moderate (10–14)
- 3 = moderately severe (15–19)
- 4 = severe (20–27)

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)

def phq9_severity_from_score(s):
    if s <= 4:  return 0
    if s <= 9:  return 1
    if s <= 14: return 2
    if s <= 19: return 3
    return 4

df["phq9_severity_class"] = df["phq9_score"].apply(phq9_severity_from_score)

print("PHQ-9")
print(f"Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")
print(f"Multi-class: {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

### 2.2 BDI-II targets

Binary cutoff: BDI-II ≥ 14:

Severity classes:
- 0 = minimal (0–13)
- 1 = mild (14–19)
- 2 = moderate (20–28)
- 3 = severe (29–63)

In [0]:
df["bdi_depressed"] = (df["bdi_score"] >= 14).astype(int)

def bdi_severity_from_score(s):
    if s <= 13: return 0
    if s <= 19: return 1
    if s <= 28: return 2
    return 3

df["bdi_severity_class"] = df["bdi_score"].apply(bdi_severity_from_score)

print("BDI-II")
print(f"Binary (>=14): {df['bdi_depressed'].value_counts().to_dict()}")
print(f"Multi-class: {df['bdi_severity_class'].value_counts().sort_index().to_dict()}")

### 2.3 Feature sets

In [0]:
ALL_FEATURES = FEATURE_COLS[:]

THEORY_FEATURES = [
    "avg_bias_neg_vs_pos",
    "avg_bias_neg_vs_neu",
    "avg_bias_pos_vs_neu",
    "avg_dwell_neg__neg_vs_pos",
    "avg_fix_prop_neg__neg_vs_pos",
    "avg_dwell_pos__pos_vs_neu",
    "avg_fix_prop_pos__pos_vs_neu",
    "avg_revisit_neg__neg_vs_pos",
    "avg_revisit_neg__neg_vs_neu",
    "first_fix_prob_neg__neg_vs_pos",
    "first_fix_prob_pos__pos_vs_neu",
    "avg_scanpath_length",
    "avg_gaze_entropy",
]

missing = [f for f in THEORY_FEATURES if f not in ALL_FEATURES]
assert not missing, f"Theory features not found in ALL_FEATURES: {missing}"

FEATURE_SETS = {
    "All Features": ALL_FEATURES,
    "Theory-Driven": THEORY_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
na_counts = df[ALL_FEATURES].isna().sum()
print("Features with NaN (top 15):")
print(na_counts[na_counts > 0].sort_values(ascending=False).head(15))
print()
print(f"Rows with any NaN: {df[ALL_FEATURES].isna().any(axis=1).sum()} / {len(df)}")

In [0]:
target_cols = [
    "phq9_score", "phq9_depressed", "phq9_severity_class",
    "bdi_score",  "bdi_depressed",  "bdi_severity_class",
]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. Binary classification

Four binary analyses combining two depression scales with two threshold-setting strategies:
- clinical cutoff (PHQ-9 ≥ 10, BDI-II ≥ 14)
- extremes task (drop the ambiguous middle; compare minimal vs severe)

### 4.1 PHQ-9 cutoff (≥ 10)

In [0]:
y_phq9_binary = df_clean["phq9_depressed"].values
phq9_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups)
print()
print(phq9_binary_df.to_string(index=False))

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups, phq9_binary_df, save_name="phq9_clinical_static_best")

### 4.2 PHQ-9 extremes (minimal vs severe)

Minimal (PHQ-9 ≤ 4) vs Severe (PHQ-9 ≥ 20)

In [0]:
df_phq9_extreme = df_clean[(df_clean["phq9_score"] <= 4) | (df_clean["phq9_score"] >= 20)].copy()
df_phq9_extreme["phq9_extreme"] = (df_phq9_extreme["phq9_score"] >= 20).astype(int)

groups_phq9_extreme = df_phq9_extreme["uid"].values

print(f"PHQ-9 extremes dataset: {len(df_phq9_extreme)} users")
print(f"Minimal (<=4):  {(df_phq9_extreme['phq9_extreme'] == 0).sum()}")
print(f"Severe  (>=20): {(df_phq9_extreme['phq9_extreme'] == 1).sum()}")
print()

y_phq9_extreme = df_phq9_extreme["phq9_extreme"].values
phq9_extreme_df = run_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme)
print()
print(phq9_extreme_df.to_string(index=False))

In [0]:
plot_best_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme, phq9_extreme_df, save_name="phq9_extreme_static_best")

### 4.3 BDI-II cutoff (≥ 14)

In [0]:
y_bdi_binary = df_clean["bdi_depressed"].values
bdi_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups)
print()
print(bdi_binary_df.to_string(index=False))

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups, bdi_binary_df, save_name="bdi_clinical_static_best")

### 4.4 BDI-II extremes (minimal vs severe)

Minimal (BDI ≤ 13) vs Severe (BDI ≥ 29). Users with intermediate scores are excluded.

In [0]:
df_bdi_extreme = df_clean[(df_clean["bdi_score"] <= 13) | (df_clean["bdi_score"] >= 29)].copy()
df_bdi_extreme["bdi_extreme"] = (df_bdi_extreme["bdi_score"] >= 29).astype(int)

groups_bdi_extreme = df_bdi_extreme["uid"].values

print(f"BDI-II extremes dataset: {len(df_bdi_extreme)} users")
print(f"Minimal (<=13): {(df_bdi_extreme['bdi_extreme'] == 0).sum()}")
print(f"Severe  (>=29): {(df_bdi_extreme['bdi_extreme'] == 1).sum()}")
print()

y_bdi_extreme = df_bdi_extreme["bdi_extreme"].values
bdi_extreme_df = run_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme)
print()
print(bdi_extreme_df.to_string(index=False))

In [0]:
plot_best_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme, bdi_extreme_df, save_name="bdi_extreme_static_best")

## 5. Multi-class classification

### 5.1 PHQ-9 (5 severity classes)

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_phq9_multi = df_clean["phq9_severity_class"].values.astype(int)
phq9_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups)
print()
print(phq9_multi_df.to_string(index=False))

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups, phq9_multi_df, PHQ9_LABELS, save_name="phq9_multiclass_static_best")

### 5.2 BDI-II (4 severity classes)

In [0]:
BDI_LABELS = ["Minimal", "Mild", "Moderate", "Severe"]
y_bdi_multi = df_clean["bdi_severity_class"].values.astype(int)
bdi_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups)
print()
print(bdi_multi_df.to_string(index=False))

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups, bdi_multi_df, BDI_LABELS, save_name="bdi_multiclass_static_best")

## 6. Regression

### 6.1 PHQ-9 score prediction

In [0]:
y_phq9_reg = df_clean["phq9_score"].values
phq9_reg_df = run_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups)
print()
print(phq9_reg_df.to_string(index=False))

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups, phq9_reg_df, score_name="PHQ-9", save_name="phq9_regression_static_best")

### 6.2 BDI-II score prediction

In [0]:
y_bdi_reg = df_clean["bdi_score"].values
bdi_reg_df = run_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups)
print()
print(bdi_reg_df.to_string(index=False))

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups, bdi_reg_df, score_name="BDI-II", save_name="bdi_regression_static_best")

## 7. PHQ-9 vs BDI-II convergent-validity comparison

Side-by-side best performance across all tasks. If the two scales produce similar numbers, the gaze-based depression signal generalizes across questionnaires.

In [0]:
phq9_results = {
    "Binary cutoff": (phq9_binary_df, "auc_roc"),
    "Binary extremes": (phq9_extreme_df, "auc_roc"),
    "Multi-class": (phq9_multi_df, "f1_weighted"),
    "Regression": (phq9_reg_df, "r2"),
}
bdi_results = {
    "Binary cutoff": (bdi_binary_df, "auc_roc"),
    "Binary extremes": (bdi_extreme_df, "auc_roc"),
    "Multi-class": (bdi_multi_df, "f1_weighted"),
    "Regression": (bdi_reg_df, "r2"),
}

comparison_df = build_comparison_table(phq9_results, bdi_results)
print(comparison_df.to_string(index=False))

## 8. Feature importance

Feature importance on the PHQ-9 binary task (Random Forest, all features). Presented once because PHQ-9 and BDI-II converge on similar predictors.

In [0]:
plot_feature_importance(df_clean, ALL_FEATURES, y_phq9_binary, title="Feature importance (PHQ-9 binary, all features)", save_name="phq9_static_feature_importance_top20")

## 9. Summary

### 9.1 PHQ-9

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(phq9_binary_df, phq9_multi_df, phq9_reg_df, feature_order, title="PHQ-9, static features", save_name="phq9_static_summary_3panel")

### 9.2 BDI-II

In [0]:
plot_summary(bdi_binary_df, bdi_multi_df, bdi_reg_df, feature_order, title="BDI-II, static features", save_name="bdi_static_summary_3panel")